# 🟢 Underwater Image Restoration – UNET_5CH

**Architecture:** U-Net  
**Input channels:** 5 (RGB + Transmission Map t(x) + Background Light B)  
**Dataset:** EUVP (Paired subsets: underwater\_dark, underwater\_imagen, underwater\_scenes)  
**Metrics:** PSNR | SSIM | CIEDE2000 | UCIQE | UIQM  

> 📌 This is the **physics-guided 5-channel** model. Compare with `unet_rgb` notebook.

---


## 1. Install dependencies

In [1]:
# ============================================================
# CELL 1 – Install dependencies
# ============================================================
#!pip install kornia scikit-image scipy opencv-python-headless lpips -q
#print("Dependencies installed!")


## 2. Imports

In [2]:
# ============================================================
# CELL 2 – Imports
# ============================================================
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import sys, json, time, random, math, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torchvision.models import vgg16, VGG16_Weights

from skimage.metrics import peak_signal_noise_ratio as psnr_sk
from skimage.metrics import structural_similarity as ssim_sk
from skimage.color import rgb2lab, deltaE_ciede2000
from scipy.ndimage import sobel, minimum_filter
import cv2
import kornia

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


PyTorch version: 2.10.0+cu128
Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


## 3. Configuration

> ⚠️ **This cell is the ONLY difference between the two notebooks:**
> - `unet_rgb`: `IN_CHANNELS=3`, `USE_PHYSICS=False`
> - `unet_5ch`: `IN_CHANNELS=5`, `USE_PHYSICS=True`


In [3]:
# ============================================================
# CELL 3 – Configuration
# ============================================================

class Config:
    # ----------------------------------------------------------
    # IDENTITY  ← đây là phần khác nhau giữa 2 notebook
    # ----------------------------------------------------------
    MODEL_NAME   = "mobilenet_unet_5ch"
    IN_CHANNELS  = 5        # 3 = RGB only  |  5 = RGB + t(x) + B
    USE_PHYSICS  = True   # False = RGB only  |  True = 5-channel

    # ----------------------------------------------------------
    # PATHS  (Kaggle)
    # ----------------------------------------------------------
    DATA_ROOT  = "/kaggle/input/datasets/pamuduranasinghe/euvp-dataset/EUVP"
    OUTPUT_DIR = ""   # filled below

    # ----------------------------------------------------------
    # DATA
    # ----------------------------------------------------------
    IMG_SIZE         = 256
    PAIRED_SUBSETS   = ["underwater_dark", "underwater_imagen", "underwater_scenes"]
    VAL_SPLIT        = 0.10   # 10% of paired train → val
    NUM_WORKERS      = 8
    PIN_MEMORY       = True

    # ----------------------------------------------------------
    # TRAINING
    # ----------------------------------------------------------
    BATCH_SIZE    = 8
    NUM_EPOCHS    = 50       # change to 50 if GPU quota is tight
    LR            = 1e-4
    WEIGHT_DECAY  = 1e-5

    # Scheduler: halve LR every SCHEDULER_STEP epochs
    SCHEDULER_STEP  = 30
    SCHEDULER_GAMMA = 0.5

    # Early stopping
    PATIENCE    = 20
    MIN_DELTA   = 1e-4

    # ----------------------------------------------------------
    # LOSS WEIGHTS  λ1·L1 + λ2·Perceptual + λ3·SSIM
    # ----------------------------------------------------------
    LAMBDA_L1          = 1
    LAMBDA_PERCEPTUAL  = 1
    LAMBDA_SSIM        = 0
    LAMBDA_GRAD       = 0

    # ----------------------------------------------------------
    # LOGGING & SAVING
    # ----------------------------------------------------------
    SAVE_EVERY = 10   # periodic checkpoint every N epochs
    LOG_EVERY  = 1    # print full metrics every N epochs
    SEED       = 42

cfg = Config()
cfg.OUTPUT_DIR = f"/kaggle/working/outputs_{cfg.MODEL_NAME}"

os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
os.makedirs(f"{cfg.OUTPUT_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{cfg.OUTPUT_DIR}/samples", exist_ok=True)

# Reproducibility
random.seed(cfg.SEED)
np.random.seed(cfg.SEED)
torch.manual_seed(cfg.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("=" * 60)
print(f"  Model name   : {cfg.MODEL_NAME}")
print(f"  Input channels: {cfg.IN_CHANNELS}")
print(f"  Use physics  : {cfg.USE_PHYSICS}")
print(f"  Epochs       : {cfg.NUM_EPOCHS}")
print(f"  Batch size   : {cfg.BATCH_SIZE}")
print(f"  Output dir   : {cfg.OUTPUT_DIR}")
print("=" * 60)


  Model name   : mobilenet_unet_5ch
  Input channels: 5
  Use physics  : True
  Epochs       : 50
  Batch size   : 8
  Output dir   : /kaggle/working/outputs_mobilenet_unet_5ch


## 4. Dataset exploration

In [4]:
# ============================================================
# CELL 4 – Dataset exploration
# ============================================================

def explore_euvp(root):
    root = Path(root)
    stats = {}

    # --- Paired subsets ---
    paired_root = root / "Paired"
    for subset in ["underwater_dark", "underwater_imagen", "underwater_scenes"]:
        sp = paired_root / subset
        if not sp.exists():
            continue
        for split in ["trainA", "trainB", "validation"]:
            d = sp / split
            if d.exists():
                imgs = [f for f in d.iterdir()
                        if f.suffix.lower() in {".jpg", ".jpeg", ".png"}]
                stats[f"Paired/{subset}/{split}"] = len(imgs)

    # --- Unpaired ---
    unpaired_root = root / "Unpaired"
    for split in ["trainA", "trainB", "validation"]:
        d = unpaired_root / split
        if d.exists():
            imgs = [f for f in d.iterdir()
                    if f.suffix.lower() in {".jpg", ".jpeg", ".png"}]
            stats[f"Unpaired/{split}"] = len(imgs)

    # --- test_samples ---
    for folder in ["GTr", "Inp"]:
        d = root / "test_samples" / folder
        if d.exists():
            imgs = [f for f in d.iterdir()
                    if f.suffix.lower() in {".jpg", ".jpeg", ".png"}]
            stats[f"test_samples/{folder}"] = len(imgs)

    return stats

stats = explore_euvp(cfg.DATA_ROOT)
print("EUVP Dataset structure:")
print("-" * 45)
for k, v in sorted(stats.items()):
    print(f"  {k:<40} {v:>5} images")
print("-" * 45)


EUVP Dataset structure:
---------------------------------------------
  Paired/underwater_dark/trainA             5550 images
  Paired/underwater_dark/trainB             5550 images
  Paired/underwater_dark/validation          570 images
  Paired/underwater_scenes/trainA           2185 images
  Paired/underwater_scenes/trainB           2185 images
  Paired/underwater_scenes/validation        130 images
  Unpaired/trainA                           3205 images
  Unpaired/trainB                           3140 images
  Unpaired/validation                        330 images
  test_samples/GTr                           515 images
  test_samples/Inp                           515 images
---------------------------------------------


## 5. Physics extraction (UDCP)

Estimates **transmission map t(x)** and **background light B** from underwater images  
using the Underwater Dark Channel Prior (UDCP) + guided image filter refinement.  
`compute_physics_maps()` is called per-sample inside the Dataset when `USE_PHYSICS=True`.


In [5]:
# ============================================================
# CELL 5 – Physics extraction (UDCP)
#           Used only when USE_PHYSICS = True
# ============================================================

def estimate_background_light(image_np, percentile=0.1):
    """
    Estimate global background light B via UDCP strategy.
    image_np : (H, W, 3) float32 in [0,1], RGB
    Returns  : B, shape (3,) float32
    """
    # In underwater images, red attenuates fastest → use G & B channels
    dark_gb = np.min(image_np[:, :, 1:], axis=2)  # (H, W)

    n_pixels  = dark_gb.size
    n_top     = max(1, int(n_pixels * percentile / 100.0))
    flat_idx  = np.argsort(dark_gb.flatten())[-n_top:]
    h_idx, w_idx = np.unravel_index(flat_idx, dark_gb.shape)

    # Among candidates pick the brightest overall pixel
    candidate_intensity = np.mean(image_np[h_idx, w_idx, :], axis=1)
    best = np.argmax(candidate_intensity)
    return image_np[h_idx[best], w_idx[best], :].astype(np.float32)


def guided_filter_np(guide, src, radius=15, eps=1e-3):
    """
    Edge-preserving guided image filter (He et al., 2013).
    guide, src : (H, W) float32
    """
    guide = guide.astype(np.float64)
    src   = src.astype(np.float64)
    r     = radius

    def box(img):
        return cv2.boxFilter(img, -1, (2*r+1, 2*r+1))

    N      = box(np.ones_like(guide))
    mI     = box(guide) / N
    mp     = box(src)   / N
    mIp    = box(guide * src) / N
    covIp  = mIp - mI * mp
    mII    = box(guide * guide) / N
    varI   = mII - mI * mI

    a = covIp / (varI + eps)
    b = mp - a * mI

    ma = box(a) / N
    mb = box(b) / N
    return (ma * guide + mb).astype(np.float32)


def estimate_transmission_udcp(image_np, B, omega=0.95, patch_size=15):
    """
    Estimate spatially varying transmission map t(x) ∈ [0.1, 1].
    image_np : (H, W, 3) float32 [0,1]
    B        : (3,) float32  background light
    """
    B_safe      = np.maximum(B, 1e-6)
    normalized  = np.clip(image_np / B_safe, 0.0, 1.0)

    # UDCP: green & blue channels
    dark        = np.min(normalized[:, :, 1:], axis=2)
    dark_ch     = minimum_filter(dark, size=patch_size)

    t_rough = np.clip(1.0 - omega * dark_ch, 0.1, 1.0)

    # Guided-filter refinement
    guide   = np.mean(image_np, axis=2).astype(np.float32)
    t_ref   = guided_filter_np(guide, t_rough, radius=15, eps=1e-3)
    return np.clip(t_ref, 0.1, 1.0)


def compute_physics_maps(image_np):
    """
    Compute (t_map, b_map) from a float32 RGB image in [0,1].
    Returns:
        t_map : (H, W) float32  – transmission map
        b_map : (H, W) float32  – spatially broadcast background light (scalar mean)
    """
    B     = estimate_background_light(image_np)
    t_map = estimate_transmission_udcp(image_np, B)
    b_map = np.full(t_map.shape, float(np.mean(B)), dtype=np.float32)
    return t_map, b_map

# Quick sanity check (CPU only, ~1 s)
_dummy = np.random.rand(256, 256, 3).astype(np.float32)
_t, _b = compute_physics_maps(_dummy)
print(f"Physics maps OK – t_map: {_t.shape}, range [{_t.min():.3f}, {_t.max():.3f}]")
print(f"                  b_map: {_b.shape}, range [{_b.min():.3f}, {_b.max():.3f}]")
del _dummy, _t, _b


Physics maps OK – t_map: (256, 256), range [0.995, 0.999]
                  b_map: (256, 256), range [0.987, 0.987]


## 6. Dataset & DataLoaders

In [6]:
# ============================================================
# CELL 6 – Dataset & DataLoaders
# ============================================================

IMG_EXTS = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}

def collect_paired_train(data_root, subsets):
    """
    Returns list of (inp_path, gt_path) from Paired/<subset>/trainA & trainB.
    Matches files by stem name.
    """
    pairs = []
    paired_root = Path(data_root) / "Paired"
    for subset in subsets:
        a_dir = paired_root / subset / "trainA"
        b_dir = paired_root / subset / "trainB"
        if not (a_dir.exists() and b_dir.exists()):
            print(f"  [WARN] Missing: {a_dir} or {b_dir}")
            continue
        b_dict = {f.stem: f for f in b_dir.iterdir() if f.suffix.lower() in IMG_EXTS}
        for a_file in sorted(a_dir.iterdir()):
            if a_file.suffix.lower() not in IMG_EXTS:
                continue
            if a_file.stem in b_dict:
                pairs.append((str(a_file), str(b_dict[a_file.stem])))
    return pairs


def collect_test_pairs(data_root):
    """Returns list of (inp_path, gt_path) from test_samples/{Inp,GTr}."""
    inp_dir = Path(data_root) / "test_samples" / "Inp"
    gt_dir  = Path(data_root) / "test_samples" / "GTr"
    pairs   = []
    if not (inp_dir.exists() and gt_dir.exists()):
        print("[WARN] test_samples not found – test set will be empty.")
        return pairs
    gt_dict = {f.stem: f for f in gt_dir.iterdir() if f.suffix.lower() in IMG_EXTS}
    for inp_file in sorted(inp_dir.iterdir()):
        if inp_file.suffix.lower() not in IMG_EXTS:
            continue
        if inp_file.stem in gt_dict:
            pairs.append((str(inp_file), str(gt_dict[inp_file.stem])))
    return pairs


class EUVPDataset(Dataset):
    """
    Paired underwater image dataset.
    When use_physics=True, __getitem__ returns a 5-channel tensor
    [R, G, B, t(x), B_map]; otherwise the standard 3-channel RGB tensor.
    """
    def __init__(self, pairs, img_size=256, augment=False, use_physics=False):
        self.pairs       = pairs
        self.img_size    = img_size
        self.augment     = augment
        self.use_physics = use_physics
        self.resize      = T.Resize((img_size, img_size), antialias=True)
        self.to_tensor   = T.ToTensor()

    def __len__(self):
        return len(self.pairs)

    def _load(self, path):
        img = Image.open(path).convert("RGB")
        return self.resize(img)

    def _augment(self, inp_pil, gt_pil):
        if random.random() > 0.5:
            inp_pil = TF.hflip(inp_pil); gt_pil = TF.hflip(gt_pil)
        if random.random() > 0.5:
            inp_pil = TF.vflip(inp_pil); gt_pil = TF.vflip(gt_pil)
        if random.random() > 0.5:
            angle   = random.uniform(-10, 10)
            inp_pil = TF.rotate(inp_pil, angle)
            gt_pil  = TF.rotate(gt_pil,  angle)
        return inp_pil, gt_pil

    def __getitem__(self, idx):
        inp_path, gt_path = self.pairs[idx]
        inp_pil  = self._load(inp_path)
        gt_pil   = self._load(gt_path)

        if self.augment:
            inp_pil, gt_pil = self._augment(inp_pil, gt_pil)

        inp_t = self.to_tensor(inp_pil)          # (3, H, W) in [0, 1]
        gt_t  = self.to_tensor(gt_pil)           # (3, H, W)

        # -------------------------------------------------------
        # Physics channels  ← ONLY when USE_PHYSICS = True
        # -------------------------------------------------------
        if self.use_physics:
            inp_np       = np.array(inp_pil, dtype=np.float32) / 255.0  # (H,W,3)
            t_map, b_map = compute_physics_maps(inp_np)                  # (H,W) each
            t_t = torch.from_numpy(t_map).unsqueeze(0)  # (1, H, W)
            b_t = torch.from_numpy(b_map).unsqueeze(0)  # (1, H, W)
            inp_t = torch.cat([inp_t, t_t, b_t], dim=0) # (5, H, W)

        return inp_t, gt_t


# ---- Build pair lists ----
all_pairs  = collect_paired_train(cfg.DATA_ROOT, cfg.PAIRED_SUBSETS)
test_pairs = collect_test_pairs(cfg.DATA_ROOT)

# ---- Train / Val split ----
random.shuffle(all_pairs)
val_n      = int(len(all_pairs) * cfg.VAL_SPLIT)
val_pairs  = all_pairs[:val_n]
train_pairs = all_pairs[val_n:]

print(f"Train pairs : {len(train_pairs):,}")
print(f"Val pairs   : {len(val_pairs):,}")
print(f"Test pairs  : {len(test_pairs):,}")

# ---- Datasets ----
train_ds = EUVPDataset(train_pairs, cfg.IMG_SIZE, augment=True,  use_physics=cfg.USE_PHYSICS)
val_ds   = EUVPDataset(val_pairs,   cfg.IMG_SIZE, augment=False, use_physics=cfg.USE_PHYSICS)
test_ds  = EUVPDataset(test_pairs,  cfg.IMG_SIZE, augment=False, use_physics=cfg.USE_PHYSICS)

# ---- DataLoaders ----
train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,
                          num_workers=cfg.NUM_WORKERS, pin_memory=cfg.PIN_MEMORY,
                          drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=cfg.BATCH_SIZE, shuffle=False,
                          num_workers=cfg.NUM_WORKERS, pin_memory=cfg.PIN_MEMORY)
test_loader  = DataLoader(test_ds,  batch_size=1,              shuffle=False,
                          num_workers=0,               pin_memory=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches  : {len(val_loader)}")
print(f"Test samples : {len(test_ds)}")

# Verify tensor shapes
_inp, _gt = train_ds[0]
print(f"Input tensor : {_inp.shape}  ← should be ({cfg.IN_CHANNELS}, {cfg.IMG_SIZE}, {cfg.IMG_SIZE})")
print(f"GT tensor    : {_gt.shape}")
del _inp, _gt


  [WARN] Missing: /kaggle/input/datasets/pamuduranasinghe/euvp-dataset/EUVP/Paired/underwater_imagen/trainA or /kaggle/input/datasets/pamuduranasinghe/euvp-dataset/EUVP/Paired/underwater_imagen/trainB
Train pairs : 6,962
Val pairs   : 773
Test pairs  : 515
Train batches: 870
Val batches  : 97
Test samples : 515
Input tensor : torch.Size([5, 256, 256])  ← should be (5, 256, 256)
GT tensor    : torch.Size([3, 256, 256])


## 7. Sample visualisation

In [7]:
# ============================================================
# CELL 7 – Visualise sample pairs
# ============================================================

def show_samples(dataset, n=4, save_path=None):
    fig, axes = plt.subplots(2, n, figsize=(4*n, 8))
    indices = random.sample(range(len(dataset)), min(n, len(dataset)))

    for col, idx in enumerate(indices):
        inp_t, gt_t = dataset[idx]
        inp_rgb = inp_t[:3].permute(1, 2, 0).numpy().clip(0, 1)
        gt_rgb  = gt_t.permute(1, 2, 0).numpy().clip(0, 1)

        axes[0, col].imshow(inp_rgb)
        axes[0, col].set_title(f"Input (degraded)\nidx={idx}")
        axes[0, col].axis("off")

        axes[1, col].imshow(gt_rgb)
        axes[1, col].set_title("Ground Truth")
        axes[1, col].axis("off")

    plt.suptitle(f"Sample pairs — {cfg.MODEL_NAME}", fontsize=14)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.show()
    plt.close()

show_samples(val_ds, n=4, save_path=f"{cfg.OUTPUT_DIR}/sample_pairs.png")


## 8. U-Net architecture

In [8]:
class MBConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, expand_ratio=2):
        super().__init__()
        mid_ch = in_ch * expand_ratio
        self.use_res = (in_ch == out_ch)
        
        self.expand = nn.Sequential(
            nn.Conv2d(in_ch, mid_ch, kernel_size=1, bias=False),
            nn.BatchNorm2d(mid_ch),
            nn.ReLU6(inplace=True)
        ) if expand_ratio != 1 else nn.Identity()
        
       
        self.depthwise = nn.Sequential(
            nn.Conv2d(mid_ch, mid_ch, kernel_size=3, padding=1, groups=mid_ch, bias=False),
            nn.BatchNorm2d(mid_ch),
            nn.ReLU6(inplace=True)
        )
        
       
        self.project = nn.Sequential(
            nn.Conv2d(mid_ch, out_ch, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_ch)
        )

    def forward(self, x):
        out = self.expand(x)
        out = self.depthwise(out)
        out = self.project(out)
        if self.use_res:
            return x + out
        return out

class MobileDoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, expand_ratio=2):
        super().__init__()
        
        self.net = nn.Sequential(
            MBConvBlock(in_ch, out_ch, expand_ratio),
            MBConvBlock(out_ch, out_ch, expand_ratio)
        )

    def forward(self, x):
        return self.net(x)

class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.MaxPool2d(2),
            MobileDoubleConv(in_ch, out_ch)
        )

    def forward(self, x):
        return self.net(x)

class Up(nn.Module):
    def __init__(self, prev_ch, skip_ch, out_ch, bilinear=True):
        super().__init__()
        if bilinear:
            self.up = nn.Sequential(
                nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True),
                
                nn.Conv2d(prev_ch, prev_ch // 2, kernel_size=1, bias=False)
            )
            combined_ch = (prev_ch // 2) + skip_ch
        else:
            self.up = nn.ConvTranspose2d(prev_ch, prev_ch // 2, kernel_size=2, stride=2)
            combined_ch = (prev_ch // 2) + skip_ch
            
        self.compress = nn.Sequential(
            nn.Conv2d(combined_ch, out_ch, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU6(inplace=True)
        )
        
        self.conv = MobileDoubleConv(out_ch, out_ch)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        dY = x2.size(2) - x1.size(2)
        dX = x2.size(3) - x1.size(3)
        x1 = F.pad(x1, [dX // 2, dX - dX // 2, dY // 2, dY - dY // 2])
        
        x = torch.cat([x2, x1], dim=1)
        x = self.compress(x)
        return self.conv(x)

class MobileUNet(nn.Module):
    def __init__(self, in_channels=5, out_channels=3, features=(64, 128, 192, 256), bilinear=True):
        super().__init__()
        f = features
        
        # Mạng Encoder
        self.enc1 = MobileDoubleConv(in_channels, f[0])
        self.enc2 = Down(f[0], f[1])
        self.enc3 = Down(f[1], f[2])
        self.enc4 = Down(f[2], f[3])
        
        # Tầng Bottleneck
        self.bottleneck = Down(f[3], f[3] * 2)
        
        # Mạng Decoder
        self.dec4 = Up(f[3] * 2, f[3], f[3], bilinear)
        self.dec3 = Up(f[3],     f[2], f[2], bilinear)
        self.dec2 = Up(f[2],     f[1], f[1], bilinear)
        self.dec1 = Up(f[1],     f[0], f[0], bilinear)
        
        # Tầng Head 
        self.head = nn.Sequential(
            nn.Conv2d(f[0], out_channels, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        bn = self.bottleneck(e4)

        d4 = self.dec4(bn, e4)
        d3 = self.dec3(d4, e3)
        d2 = self.dec2(d3, e2)
        d1 = self.dec1(d2, e1)

        return self.head(d1)

model = MobileUNet(in_channels=cfg.IN_CHANNELS, out_channels=3).to(device)

# ── Cài thop nếu chưa có ────────────────────────────────────────
try:
    from thop import profile, clever_format
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "thop", "-q"])
    from thop import profile, clever_format

def model_profile(model, in_channels, img_size, device, n_runs=200):
    m = model.module if isinstance(model, nn.DataParallel) else model
    m.eval()
    dummy = torch.randn(1, in_channels, img_size, img_size, device=device)

    # ── Params & FLOPs ───────────────────────────────────────────
    with torch.no_grad():
        flops, params = profile(m, inputs=(dummy,), verbose=False)
    gflops  = flops  / 1e9
    mparams = params / 1e6

    # ── Inference time (GPU warmup trước) ────────────────────────
    with torch.no_grad():
        for _ in range(20):                        # warmup
            _ = m(dummy)
        if device.type == "cuda":
            torch.cuda.synchronize()

        import time
        t0 = time.perf_counter()
        for _ in range(n_runs):
            _ = m(dummy)
        if device.type == "cuda":
            torch.cuda.synchronize()
        ms_per_img = (time.perf_counter() - t0) / n_runs * 1000

    print("=" * 48)
    print(f"  Model      : {m.__class__.__name__}")
    print(f"  Parameters : {mparams:>8.3f} M")
    print(f"  FLOPs      : {gflops:>8.3f} G")
    print(f"  Inference  : {ms_per_img:>8.3f} ms / image  (batch=1)")
    print("=" * 48)
    return {"params_M": mparams, "gflops": gflops, "ms_per_img": ms_per_img}


profile_stats = model_profile(model, cfg.IN_CHANNELS, cfg.IMG_SIZE, device)

# Quick forward pass check
with torch.no_grad():
    _x = torch.randn(2, cfg.IN_CHANNELS, cfg.IMG_SIZE, cfg.IMG_SIZE).to(device)
    _y = model(_x)
    print(f"Forward OK: input {_x.shape} → output {_y.shape}")
del _x, _y

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 101.8 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

  Model      : MobileUNet
  Parameters :    3.704 M
  FLOPs      :   14.019 G
  Inference  :   20.369 ms / image  (batch=1)
Forward OK: input torch.Size([2, 5, 256, 256]) → output torch.Size([2, 3, 256, 256])


## 9. Loss functions  (L1 + VGG-Perceptual + SSIM)

In [9]:
# ============================================================
# CELL 9 – Loss Functions (FIXED VGG PERCEPTUAL LOSS)
# ============================================================

class SobelEdgeFilter(nn.Module):
    def __init__(self):
        super().__init__()
        # Sobel kernels — fixed weights, không train
        kx = torch.tensor([[-1.,  0.,  1.],
                            [-2.,  0.,  2.],
                            [-1.,  0.,  1.]]).view(1, 1, 3, 3)
        ky = torch.tensor([[-1., -2., -1.],
                            [ 0.,  0.,  0.],
                            [ 1.,  2.,  1.]]).view(1, 1, 3, 3)
        self.register_buffer('kx', kx)
        self.register_buffer('ky', ky)

    def forward(self, x):
        # x: (B, 3, H, W) — chuyển sang grayscale
        gray = (0.2989 * x[:, 0:1]
               + 0.5870 * x[:, 1:2]
               + 0.1140 * x[:, 2:3])          # (B, 1, H, W)

        gx = F.conv2d(gray, self.kx, padding=1)  # (B, 1, H, W)
        gy = F.conv2d(gray, self.ky, padding=1)  # (B, 1, H, W)

        edge = torch.sqrt(gx ** 2 + gy ** 2 + 1e-8)   # magnitude
        # Normalize per-image về [0,1] để ổn định
        b = edge.size(0)
        mn = edge.view(b, -1).min(dim=1)[0].view(b, 1, 1, 1)
        mx = edge.view(b, -1).max(dim=1)[0].view(b, 1, 1, 1)
        edge = (edge - mn) / (mx - mn + 1e-8)
        return edge                               # (B, 1, H, W)


class LatentStructuralGradientLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.sobel = SobelEdgeFilter()

    def forward(self, pred, target):
        # Bước 1: Sobel edge maps
        E_pred   = self.sobel(pred)    # (B, 1, H, W)
        E_target = self.sobel(target)  # (B, 1, H, W)

        # Bước 2a: first-order horizontal differences (W axis) 
        grad_h_pred   = torch.diff(E_pred, dim=3)
        grad_h_target = torch.diff(E_target, dim=3)

        # Bước 2b: first-order vertical differences (H axis)
        grad_v_pred   = torch.diff(E_pred, dim=2)
        grad_v_target = torch.diff(E_target, dim=2)

        # Bước 3: MSE
        loss_h = F.mse_loss(grad_h_pred, grad_h_target)
        loss_v = F.mse_loss(grad_v_pred, grad_v_target)

        return (loss_h + loss_v) * 0.5

class VGGPerceptualLoss(nn.Module):
    """VGG-16 feature-space loss (relu1_2 + relu2_2 layers)."""
    def __init__(self, device):
        super().__init__()
        vgg    = vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features
        self.s1 = nn.Sequential(*list(vgg.children())[:4]).to(device).eval()
        self.s2 = nn.Sequential(*list(vgg.children())[4:9]).to(device).eval()
        for p in self.parameters():
            p.requires_grad = False

    def forward(self, pred, target):
        p1, t1 = self.s1(pred), self.s1(target)
        # LỖI ĐÃ ĐƯỢC FIX Ở ĐÂY: Truyền p1, t1 vào s2 thay vì truyền thẳng pred, target
        p2, t2 = self.s2(p1), self.s2(t1)
        return F.l1_loss(p1, t1) + F.l1_loss(p2, t2)

class SSIMLoss(nn.Module):
    def __init__(self, window_size=11):
        super().__init__()
        self.ws = window_size

    def forward(self, pred, target):
        # kornia.metrics.ssim returns SSIM map; mean gives scalar SSIM
        ssim_map = kornia.metrics.ssim(pred, target, self.ws)
        return 1.0 - ssim_map.mean()

class CompositeLoss(nn.Module):
    def __init__(self, λ_l1=1.0, λ_perc=1.0, λ_ssim=0, λ_gd=0, device="cpu"):
        super().__init__()
        self.λ_l1   = λ_l1
        self.λ_perc = λ_perc
        self.λ_ssim = λ_ssim
        self.λ_gd = λ_gd
        self.l1   = nn.L1Loss()
        self.perc = VGGPerceptualLoss(device)
        self.ssim = SSIMLoss()
        self.grad = LatentStructuralGradientLoss()

    def forward(self, pred, target):
        l_l1   = self.l1(pred, target)
        l_perc = self.perc(pred, target)
        l_ssim = self.ssim(pred, target)
        l_gd   = self.grad(pred, target)
        total  = self.λ_l1 * l_l1 + self.λ_perc * l_perc + self.λ_ssim * l_ssim + self.λ_gd * l_gd
        return total, {
            "l1":          l_l1.item(),
            "perceptual":  l_perc.item(),
            "ssim_loss":   l_ssim.item(),
            "gradient_loss": l_gd.item(),
            "total":       total.item(),
        }

criterion = CompositeLoss(
    λ_l1   = cfg.LAMBDA_L1,
    λ_perc = cfg.LAMBDA_PERCEPTUAL,
    λ_ssim = cfg.LAMBDA_SSIM,
    λ_gd   = cfg.LAMBDA_GRAD,
).to(device)

print("Criterion: L1 + VGG-Perceptual + SSIM + Gradient")
print(f"  λ_L1={cfg.LAMBDA_L1}  λ_Perceptual={cfg.LAMBDA_PERCEPTUAL}  λ_SSIM={cfg.LAMBDA_SSIM}  λ_GD={cfg.LAMBDA_GRAD}")

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 210MB/s]


Criterion: L1 + VGG-Perceptual + SSIM + Gradient
  λ_L1=1  λ_Perceptual=1  λ_SSIM=0  λ_GD=0


## 10. Optimizer & Scheduler

In [10]:
# ============================================================
# CELL 10 – Optimizer & Scheduler
# ============================================================

optimizer = torch.optim.Adam(
    model.parameters(),
    lr           = cfg.LR,
    weight_decay = cfg.WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size = cfg.SCHEDULER_STEP,
    gamma     = cfg.SCHEDULER_GAMMA,
)
print(f"Optimizer : Adam  (lr={cfg.LR}, wd={cfg.WEIGHT_DECAY})")
print(f"Scheduler : StepLR  (step={cfg.SCHEDULER_STEP}, γ={cfg.SCHEDULER_GAMMA})")


Optimizer : Adam  (lr=0.0001, wd=1e-05)
Scheduler : StepLR  (step=30, γ=0.5)


## 11. Evaluation metrics

In [11]:
# ============================================================
# CELL 11 – Evaluation Metrics
# ============================================================

def compute_psnr(pred, target):
    """pred, target: numpy (H,W,3) float [0,1]"""
    return float(psnr_sk(target, pred, data_range=1.0))


def compute_ssim(pred, target):
    """pred, target: numpy (H,W,3) float [0,1]"""
    return float(ssim_sk(target, pred, data_range=1.0, channel_axis=2))


def compute_ciede2000(pred, target):
    """Mean CIEDE2000 colour difference (lower = better)."""
    pred_lab   = rgb2lab(pred.clip(0, 1))
    target_lab = rgb2lab(target.clip(0, 1))
    return float(np.mean(deltaE_ciede2000(target_lab, pred_lab)))


def compute_uciqe(img):
    """
    UCIQE – Yang & Sowmya (2015).
    img: (H,W,3) float [0,1] RGB
    Fix: chuyển OpenCV LAB uint8 → chuẩn CIELab trước khi tính.
    """
    u8 = (img.clip(0, 1) * 255).astype(np.uint8)

    # ── OpenCV LAB uint8 → standard CIELab ──────────────────────
    # OpenCV:  L ∈ [0,255],  A ∈ [0,255],  B ∈ [0,255]
    # Standard: L ∈ [0,100], a ∈ [-128,127], b ∈ [-128,127]
    lab   = cv2.cvtColor(u8, cv2.COLOR_RGB2LAB).astype(np.float64)
    L_std = lab[:, :, 0] * (100.0 / 255.0)   # L: 0 → 100
    a_std = lab[:, :, 1] - 128.0              # a: -128 → 127
    b_std = lab[:, :, 2] - 128.0             # b: -128 → 127

    # σc – std của chroma
    chroma  = np.sqrt(a_std ** 2 + b_std ** 2)
    sigma_c = np.std(chroma)

    # con_l – contrast của L (top 1% − bottom 1%), trong [0,100]
    sorted_L = np.sort(L_std.flatten())
    n        = len(sorted_L)
    con_l    = (np.mean(sorted_L[int(0.99 * n):])
              - np.mean(sorted_L[:max(1, int(0.01 * n))]))

    # μs – mean saturation HSV ∈ [0,1]
    hsv  = cv2.cvtColor(u8, cv2.COLOR_RGB2HSV).astype(np.float64)
    mu_s = np.mean(hsv[:, :, 1]) / 255.0

    return float(0.4680 * sigma_c + 0.2745 * con_l + 0.2576 * mu_s)


def compute_uiqm(image):
    """
    UIQM (Panetta et al., 2016).
    image : (H,W,3) float [0,1] RGB
    Returns scalar (higher = better)
    """
    img = image.clip(0, 1).astype(np.float64)
    R, G, B = img[:,:,0], img[:,:,1], img[:,:,2]

    # UICM – colorfulness
    RG = R - G
    YB = 0.5*(R + G) - B
    uicm = (-0.0268 * np.sqrt(np.mean(RG)**2 + np.mean(YB)**2)
             + 0.1586 * np.sqrt(np.std(RG)**2  + np.std(YB)**2))

    # UISM – sharpness (Sobel on each channel)
    gray = 0.2989*R + 0.5870*G + 0.1140*B
    sx   = sobel(gray, axis=0)
    sy   = sobel(gray, axis=1)
    uism = np.mean(np.sqrt(sx**2 + sy**2))

    # UIConM – contrast
    mu_g    = np.mean(gray)
    sigma_g = np.std(gray)
    uiconm  = sigma_g / (mu_g + 1e-8)

    c1, c2, c3 = 0.0282, 0.2953, 3.5753
    return float(c1*uicm + c2*uism + c3*uiconm)


@torch.no_grad()
def evaluate_loader(model, loader, device, max_samples=None, desc="Evaluating"):
    """Compute full metric suite on a DataLoader."""
    model.eval()
    results = {k: [] for k in ("psnr", "ssim", "ciede2000", "uciqe", "uiqm")}
    count   = 0

    for inp, gt in loader:
        inp  = inp.to(device)
        pred = model(inp)

        pred_np = pred.cpu().permute(0,2,3,1).numpy().clip(0, 1)
        gt_np   = gt.permute(0,2,3,1).numpy().clip(0, 1)

        for i in range(pred_np.shape[0]):
            p, g = pred_np[i], gt_np[i]
            results["psnr"].append(compute_psnr(p, g))
            results["ssim"].append(compute_ssim(p, g))
            results["ciede2000"].append(compute_ciede2000(p, g))
            results["uciqe"].append(compute_uciqe(p))
            results["uiqm"].append(compute_uiqm(p))
            count += 1
            if max_samples and count >= max_samples:
                break
        if max_samples and count >= max_samples:
            break

    return {k: float(np.mean(v)) for k, v in results.items()}, count

print("Metrics defined: PSNR | SSIM | CIEDE2000 | UCIQE | UIQM")


Metrics defined: PSNR | SSIM | CIEDE2000 | UCIQE | UIQM


## 12. Training loop

Features:
- Prints loss + metrics every `LOG_EVERY` epochs
- Saves **best model immediately** when PSNR improves
- Periodic checkpoint every `SAVE_EVERY` epochs
- Early stopping with `PATIENCE` epochs


In [12]:
# ============================================================
# CELL 12 – Training Loop
# ============================================================

# ---- Early Stopping ----
class EarlyStopping:
    def __init__(self, patience=15, min_delta=1e-4, mode="max"):
        self.patience   = patience
        self.min_delta  = min_delta
        self.mode       = mode
        self.counter    = 0
        self.best       = None
        self.stop       = False

    def __call__(self, score):
        if self.best is None:
            self.best = score
        elif (self.mode == "max" and score > self.best + self.min_delta) or \
             (self.mode == "min" and score < self.best - self.min_delta):
            self.best    = score
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


# ---- Checkpoint helpers ----
def save_ckpt(model, optimizer, epoch, metrics, path):
    torch.save({
        "epoch":     epoch,
        "model":     model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "metrics":   metrics,
    }, path)


def load_ckpt(path, model, optimizer=None, device="cpu"):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model"])
    if optimizer and "optimizer" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer"])
    return ckpt["epoch"], ckpt.get("metrics", {})


# ---- One epoch helpers ----
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    tot_loss = 0.0
    comps    = {"l1": 0.0, "perceptual": 0.0, "ssim_loss": 0.0}

    for inp, gt in loader:
        inp, gt = inp.to(device), gt.to(device)
        optimizer.zero_grad(set_to_none=True)
        pred         = model(inp)
        loss, parts  = criterion(pred, gt)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        tot_loss += loss.item()
        for k in comps:
            comps[k] += parts.get(k, 0.0)

    n = len(loader)
    return tot_loss / n, {k: v / n for k, v in comps.items()}


@torch.no_grad()
def val_loss_epoch(model, loader, criterion, device):
    model.eval()
    tot = 0.0
    for inp, gt in loader:
        inp, gt = inp.to(device), gt.to(device)
        pred    = model(inp)
        loss, _ = criterion(pred, gt)
        tot    += loss.item()
    return tot / len(loader)


# ---- History & helpers ----
history = {
    "train_loss": [], "val_loss": [],
    "val_psnr":   [], "val_ssim": [],
    "lr":         [],
}

best_psnr  = 0.0
best_epoch = 0
es         = EarlyStopping(patience=cfg.PATIENCE, min_delta=cfg.MIN_DELTA, mode="max")

BEST_PATH = f"{cfg.OUTPUT_DIR}/checkpoints/best_model.pth"
LAST_PATH = f"{cfg.OUTPUT_DIR}/checkpoints/last_model.pth"

print(f"\n{'='*65}")
print(f"  Training: {cfg.MODEL_NAME}   epochs={cfg.NUM_EPOCHS}   batch={cfg.BATCH_SIZE}")
print(f"{'='*65}")
print(f"{'Epoch':>6}  {'TrainL':>8}  {'ValL':>8}  {'PSNR':>7}  {'SSIM':>7}  {'LR':>9}  {'Time':>6}")
print("-"*65)

train_start = time.time()

for epoch in range(1, cfg.NUM_EPOCHS + 1):
    t0 = time.time()

    # --- Train ---
    tr_loss, tr_comps = train_epoch(model, train_loader, optimizer, criterion, device)

    # --- Val loss (every epoch) ---
    vl_loss = val_loss_epoch(model, val_loader, criterion, device)

    # --- Val metrics (every LOG_EVERY epochs or epoch 1) ---
    if epoch == 1 or epoch % cfg.LOG_EVERY == 0:
        val_metrics, _ = evaluate_loader(model, val_loader, device, max_samples=100)
        val_psnr = val_metrics["psnr"]
        val_ssim = val_metrics["ssim"]
    else:
        val_psnr = history["val_psnr"][-1] if history["val_psnr"] else 0.0
        val_ssim = history["val_ssim"][-1] if history["val_ssim"] else 0.0

    scheduler.step()
    cur_lr = optimizer.param_groups[0]["lr"]
    elapsed = time.time() - t0

    # --- Record ---
    history["train_loss"].append(tr_loss)
    history["val_loss"].append(vl_loss)
    history["val_psnr"].append(val_psnr)
    history["val_ssim"].append(val_ssim)
    history["lr"].append(cur_lr)

    # --- Save best immediately when PSNR improves ---
    if val_psnr > best_psnr + cfg.MIN_DELTA:
        best_psnr  = val_psnr
        best_epoch = epoch
        save_ckpt(model, optimizer, epoch,
                  {"psnr": val_psnr, "ssim": val_ssim, "val_loss": vl_loss},
                  BEST_PATH)
        flag = "  ← BEST ✓"
    else:
        flag = ""

    # --- Periodic checkpoint ---
    if epoch % cfg.SAVE_EVERY == 0:
        save_ckpt(model, optimizer, epoch,
                  {"psnr": val_psnr, "ssim": val_ssim},
                  f"{cfg.OUTPUT_DIR}/checkpoints/epoch_{epoch:04d}.pth")

    # --- Log line (every LOG_EVERY or on new best) ---
    if epoch == 1 or epoch % cfg.LOG_EVERY == 0 or flag:
        print(f"{epoch:>6}  {tr_loss:>8.4f}  {vl_loss:>8.4f}  "
              f"{val_psnr:>7.3f}  {val_ssim:>7.4f}  {cur_lr:>9.2e}  "
              f"{elapsed:>5.1f}s{flag}")

    # --- Early stopping ---
    if es(val_psnr):
        print(f"\nEarly stopping at epoch {epoch}  (no improvement for {cfg.PATIENCE} evals)")
        break

# Save last model
save_ckpt(model, optimizer, epoch, {"psnr": val_psnr, "ssim": val_ssim}, LAST_PATH)

total_min = (time.time() - train_start) / 60
print(f"\n{'='*65}")
print(f"  Done.  Best PSNR: {best_psnr:.4f} dB at epoch {best_epoch}")
print(f"  Total training time: {total_min:.1f} min")
print(f"{'='*65}")

# Save history JSON
with open(f"{cfg.OUTPUT_DIR}/training_history.json", "w") as f:
    json.dump(history, f, indent=2)
print("History saved →", f"{cfg.OUTPUT_DIR}/training_history.json")



  Training: mobilenet_unet_5ch   epochs=50   batch=8
 Epoch    TrainL      ValL     PSNR     SSIM         LR    Time
-----------------------------------------------------------------
     1    0.4710    0.3585   21.159   0.8503   1.00e-04  705.2s  ← BEST ✓
     2    0.3640    0.3322   21.574   0.8623   1.00e-04  706.4s  ← BEST ✓
     3    0.3440    0.3180   22.038   0.8755   1.00e-04  706.5s  ← BEST ✓
     4    0.3320    0.3098   22.079   0.8747   1.00e-04  705.2s  ← BEST ✓
     5    0.3230    0.3024   22.411   0.8818   1.00e-04  705.8s  ← BEST ✓
     6    0.3163    0.2974   22.576   0.8831   1.00e-04  706.3s  ← BEST ✓
     7    0.3104    0.2941   22.648   0.8863   1.00e-04  706.0s  ← BEST ✓
     8    0.3060    0.2932   22.695   0.8865   1.00e-04  705.8s  ← BEST ✓
     9    0.3020    0.2885   22.860   0.8876   1.00e-04  706.2s  ← BEST ✓
    10    0.2994    0.2888   22.782   0.8853   1.00e-04  706.1s
    11    0.2966    0.2855   22.805   0.8892   1.00e-04  706.0s
    12    0.2944    0.

## 13. Training curves

In [13]:
# ============================================================
# CELL 13 – Plot Training Curves
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

epochs_x = range(1, len(history["train_loss"]) + 1)

# Loss
axes[0].plot(epochs_x, history["train_loss"], label="Train Loss", color="steelblue")
axes[0].plot(epochs_x, history["val_loss"],   label="Val Loss",   color="tomato")
axes[0].axvline(best_epoch, linestyle="--", color="green", alpha=0.7, label=f"Best epoch={best_epoch}")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend(); axes[0].grid(True)

# PSNR
axes[1].plot(epochs_x, history["val_psnr"], color="darkorange")
axes[1].axvline(best_epoch, linestyle="--", color="green", alpha=0.7)
axes[1].axhline(best_psnr,  linestyle=":",  color="green", alpha=0.7, label=f"Best={best_psnr:.2f} dB")
axes[1].set_title("Val PSNR (dB)"); axes[1].set_xlabel("Epoch"); axes[1].legend(); axes[1].grid(True)

# SSIM
axes[2].plot(epochs_x, history["val_ssim"], color="mediumpurple")
axes[2].axvline(best_epoch, linestyle="--", color="green", alpha=0.7)
axes[2].set_title("Val SSIM"); axes[2].set_xlabel("Epoch"); axes[2].grid(True)

plt.suptitle(f"Training curves – {cfg.MODEL_NAME}", fontsize=14)
plt.tight_layout()
plt.savefig(f"{cfg.OUTPUT_DIR}/training_curves.png", dpi=120, bbox_inches="tight")
plt.show(); plt.close()
print("Saved →", f"{cfg.OUTPUT_DIR}/training_curves.png")


Saved → /kaggle/working/outputs_mobilenet_unet_5ch/training_curves.png


## 14. Test-set evaluation

In [14]:
# ============================================================
# CELL 14 – Load Best Model & Evaluate on Test Set
# ============================================================

# Load best checkpoint
ckpt_epoch, ckpt_metrics = load_ckpt(BEST_PATH, model, device=device)
print(f"Loaded best checkpoint: epoch={ckpt_epoch}  stored metrics={ckpt_metrics}")

# Full test evaluation
test_metrics, n_test = evaluate_loader(model, test_loader, device, desc="Testing")

print(f"\n{'='*55}")
print(f"  TEST RESULTS – {cfg.MODEL_NAME}  (n={n_test})")
print(f"{'='*55}")
print(f"  PSNR      : {test_metrics['psnr']:>8.4f}  dB")
print(f"  SSIM      : {test_metrics['ssim']:>8.4f}")
print(f"  CIEDE2000 : {test_metrics['ciede2000']:>8.4f}  (lower=better)")
print(f"  UCIQE     : {test_metrics['uciqe']:>8.4f}  (higher=better)")
print(f"  UIQM      : {test_metrics['uiqm']:>8.4f}  (higher=better)")
print(f"{'='*55}")

# Save results
results_dict = {
    "model_name": cfg.MODEL_NAME,
    "in_channels": cfg.IN_CHANNELS,
    "use_physics": cfg.USE_PHYSICS,
    "best_epoch": ckpt_epoch,
    "test_metrics": test_metrics,
}
with open(f"{cfg.OUTPUT_DIR}/test_results.json", "w") as f:
    json.dump(results_dict, f, indent=2)
print("Results saved →", f"{cfg.OUTPUT_DIR}/test_results.json")


Loaded best checkpoint: epoch=47  stored metrics={'psnr': 23.57669506811499, 'ssim': 0.9008517909049988, 'val_loss': 0.26677452749812725}

  TEST RESULTS – mobilenet_unet_5ch  (n=515)
  PSNR      :  26.4461  dB
  SSIM      :   0.8714
  CIEDE2000 :   5.7566  (lower=better)
  UCIQE     :  26.6519  (higher=better)
  UIQM      :   1.8152  (higher=better)
Results saved → /kaggle/working/outputs_mobilenet_unet_5ch/test_results.json


## 15. Visual results

In [15]:
# ============================================================
# CELL 15 – Visualise Restoration Results
# ============================================================

@torch.no_grad()
def visualise_results(model, dataset, n=6, save_path=None):
    model.eval()
    indices = random.sample(range(len(dataset)), min(n, len(dataset)))
    fig, axes = plt.subplots(3, n, figsize=(4*n, 12))

    for col, idx in enumerate(indices):
        inp_t, gt_t = dataset[idx]
        inp_dev  = inp_t.unsqueeze(0).to(device)
        pred_t   = model(inp_dev).squeeze(0).cpu()

        inp_rgb  = inp_t[:3].permute(1,2,0).numpy().clip(0,1)
        pred_rgb = pred_t.permute(1,2,0).numpy().clip(0,1)
        gt_rgb   = gt_t.permute(1,2,0).numpy().clip(0,1)

        psnr_v = compute_psnr(pred_rgb, gt_rgb)
        ssim_v = compute_ssim(pred_rgb, gt_rgb)

        axes[0, col].imshow(inp_rgb);  axes[0, col].set_title("Input");        axes[0, col].axis("off")
        axes[1, col].imshow(pred_rgb); axes[1, col].set_title(
            f"Restored\nPSNR={psnr_v:.2f} dB\nSSIM={ssim_v:.4f}"); axes[1, col].axis("off")
        axes[2, col].imshow(gt_rgb);   axes[2, col].set_title("Ground Truth"); axes[2, col].axis("off")

    plt.suptitle(f"Restoration results – {cfg.MODEL_NAME}", fontsize=14)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.show(); plt.close()

visualise_results(model, test_ds, n=6,
                  save_path=f"{cfg.OUTPUT_DIR}/test_visual.png")
print("Saved →", f"{cfg.OUTPUT_DIR}/test_visual.png")


Saved → /kaggle/working/outputs_mobilenet_unet_5ch/test_visual.png


## 16. Ablation / comparison table

Run this cell in **either** notebook after downloading `test_results.json` from both.  
Edit the two file paths to point to the downloaded JSON files.


In [16]:
# ============================================================
# CELL 16 – Ablation: compare 3-ch vs 5-ch on shared test set
#
# This cell can be run in EITHER notebook once you have the
# test_results.json from both runs (download from Kaggle outputs).
# Paste the paths to both JSON files below.
# ============================================================

import json, os

# ---- EDIT THESE PATHS ----
PATH_RGB = "/kaggle/working/outputs_unet_rgb/test_results.json"
PATH_5CH = "/kaggle/working/outputs_unet_5ch/test_results.json"
# --------------------------

results = {}
for name, path in [("RGB-only (3ch)", PATH_RGB), ("Physics-guided (5ch)", PATH_5CH)]:
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        results[name] = data.get("test_metrics", {})
    else:
        print(f"[WARN] Not found: {path}")

if results:
    print(f"\n{'Model':<25} {'PSNR':>8} {'SSIM':>8} {'CIEDE2000':>10} {'UCIQE':>8} {'UIQM':>8}")
    print("-"*75)
    for model_name, m in results.items():
        print(f"{model_name:<25} "
              f"{m.get('psnr',0):>8.4f} "
              f"{m.get('ssim',0):>8.4f} "
              f"{m.get('ciede2000',0):>10.4f} "
              f"{m.get('uciqe',0):>8.4f} "
              f"{m.get('uiqm',0):>8.4f}")
    print("-"*75)

    # --- Bar chart comparison ---
    metrics_keys   = ["psnr", "ssim", "uciqe", "uiqm"]
    metrics_labels = ["PSNR (dB)", "SSIM", "UCIQE", "UIQM"]
    model_names    = list(results.keys())
    n_models       = len(model_names)

    fig, axes = plt.subplots(1, len(metrics_keys), figsize=(16, 5))
    colors = ["#4C72B0", "#DD8452"]

    for ax, mk, ml in zip(axes, metrics_keys, metrics_labels):
        vals = [results[mn].get(mk, 0) for mn in model_names]
        bars = ax.bar(model_names, vals, color=colors[:n_models], edgecolor="black", width=0.5)
        ax.bar_label(bars, fmt="%.4f", padding=3, fontsize=9)
        ax.set_title(ml); ax.set_ylim(0, max(vals) * 1.15)
        ax.tick_params(axis="x", rotation=10)
        ax.grid(axis="y", alpha=0.4)

    plt.suptitle("Ablation: RGB-only vs Physics-guided (5-channel)", fontsize=13)
    plt.tight_layout()
    plt.savefig("/kaggle/working/ablation_comparison.png", dpi=120, bbox_inches="tight")
    plt.show(); plt.close()
    print("Saved → /kaggle/working/ablation_comparison.png")


[WARN] Not found: /kaggle/working/outputs_unet_rgb/test_results.json
[WARN] Not found: /kaggle/working/outputs_unet_5ch/test_results.json


## 17. Model summary & output files

In [17]:
# ============================================================
# CELL 17 – Save final model summary
# ============================================================

summary = {
    "model_name":    cfg.MODEL_NAME,
    "in_channels":   cfg.IN_CHANNELS,
    "use_physics":   cfg.USE_PHYSICS,
    "img_size":      cfg.IMG_SIZE,
    "epochs_trained": len(history["train_loss"]),
    "best_epoch":    best_epoch,
    "best_val_psnr": best_psnr,
    "total_params":  sum(p.numel() for p in model.parameters()),
    "config": {
        "batch_size":  cfg.BATCH_SIZE,
        "lr":          cfg.LR,
        "lambda_l1":   cfg.LAMBDA_L1,
        "lambda_perc": cfg.LAMBDA_PERCEPTUAL,
        "lambda_ssim": cfg.LAMBDA_SSIM,
        "patience":    cfg.PATIENCE,
    },
}
with open(f"{cfg.OUTPUT_DIR}/model_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Final model summary:")
for k, v in summary.items():
    if k != "config":
        print(f"  {k:<22}: {v}")
print("  config:")
for k, v in summary["config"].items():
    print(f"    {k:<20}: {v}")
print(f"\nAll outputs saved in: {cfg.OUTPUT_DIR}")
print("Files to download:")
for f in sorted(Path(cfg.OUTPUT_DIR).rglob("*")):
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f"  {f}  ({size_kb:.1f} KB)")


Final model summary:
  model_name            : mobilenet_unet_5ch
  in_channels           : 5
  use_physics           : True
  img_size              : 256
  epochs_trained        : 50
  best_epoch            : 47
  best_val_psnr         : 23.57669506811499
  total_params          : 3704055
  config:
    batch_size          : 8
    lr                  : 0.0001
    lambda_l1           : 1
    lambda_perc         : 1
    lambda_ssim         : 0
    patience            : 20

All outputs saved in: /kaggle/working/outputs_mobilenet_unet_5ch
Files to download:
  /kaggle/working/outputs_mobilenet_unet_5ch/checkpoints/best_model.pth  (43832.2 KB)
  /kaggle/working/outputs_mobilenet_unet_5ch/checkpoints/epoch_0010.pth  (43832.1 KB)
  /kaggle/working/outputs_mobilenet_unet_5ch/checkpoints/epoch_0020.pth  (43832.1 KB)
  /kaggle/working/outputs_mobilenet_unet_5ch/checkpoints/epoch_0030.pth  (43832.1 KB)
  /kaggle/working/outputs_mobilenet_unet_5ch/checkpoints/epoch_0040.pth  (43832.1 KB)
  /kaggle/